# Chapter 3 — Evidence, Truth, and Verifiability

**Book alignment:** Hallucination From First Principles, Chapter 3

**Question this notebook isolates:** Does comparing unresolved strings produce a resolution disagreement while normalized propositions verify correctly?

Synthetic fixtures in this notebook demonstrate the mechanism type only and do not reproduce the book's 10k-row empirical run.


In [ ]:
import numpy as np

rng = np.random.default_rng(2)


## 1. Raw strings confuse resolution with truth

Chapter 3 (sec. 1) requires normalizing a sentence into subject / relation / object / time / scope before judging truth: *acquired in 2024* can mean announcement or completion. We show one raw string carrying two readings where raw-string similarity cannot decide, while proposition comparison verifies correctly.


In [ ]:
raw = "Acme acquired Orion in 2024"
evidence_sentence = "On March 14 2024 Acme completed its acquisition of Orion"
evidence = {"subj": "Acme", "rel": "completed_acquisition", "obj": "Orion", "time": "2024-03-14"}
reading_announced = {"subj": "Acme", "rel": "announced_acquisition", "obj": "Orion", "time": "2024"}
reading_closed = {"subj": "Acme", "rel": "completed_acquisition", "obj": "Orion", "time": "2024"}


def raw_jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb)


def time_covers(ev_time, claim_time):
    return ev_time.startswith(claim_time)  # '2024-03-14' is within scope '2024'


def verify(norm_claim, norm_ev):
    if norm_claim["subj"] != norm_ev["subj"] or norm_claim["obj"] != norm_ev["obj"]:
        return "REFUTES"
    if norm_claim["rel"] != norm_ev["rel"]:
        return "INSUFFICIENT"  # announcement evidence cannot license completion and vice versa
    return "SUPPORTS" if time_covers(norm_ev["time"], norm_claim["time"]) else "INSUFFICIENT"


rj = raw_jaccard(raw, evidence_sentence)
v_announced = verify(reading_announced, evidence)
v_closed = verify(reading_closed, evidence)
print("raw string:              ", repr(raw))
print("raw jaccard vs evidence: ", round(rj, 3))
print("reading 'announced' ->", v_announced)
print("reading 'completed' ->", v_closed)


In [ ]:
# One surface string, two propositions, two verdicts: resolution decides, not the raw text.
assert v_closed == "SUPPORTS"
assert v_announced == "INSUFFICIENT"
assert v_closed != v_announced  # raw-string comparison has no way to express this split
print("resolution disagreement confirmed: identical raw string verifies differently per reading")


## 2. The attribution chain fails one link at a time

Chapter 3 (sec. 7) expands one citation into a chain: source resolution, identity, passage localization, claim-passage support, and policy admissibility. We run five synthetic citation records through stub chain checks and show each fails at a different link, including the book's 37%-vs-17% strength inflation.


In [ ]:
def check_chain(c):
    """Stub attribution chain. Each stage returns PASS / FAIL."""
    resolution = "PASS" if c["resolves"] else "FAIL"
    identity = "PASS" if (c["resolves"] and c["identity_ok"]) else ("FAIL" if c["resolves"] else "SKIP")
    localization = "PASS" if (identity == "PASS" and c["passage_ok"]) else (("FAIL" if identity == "PASS" else "SKIP"))
    support = "PASS" if (localization == "PASS" and c["support_ok"]) else (("FAIL" if localization == "PASS" else "SKIP"))
    admissible = "PASS" if (support == "PASS" and c["policy_ok"]) else (("FAIL" if support == "PASS" else "SKIP"))
    return [resolution, identity, localization, support, admissible]


stages = ["resolve", "identity", "localize", "support", "admit"]
cites = [
    {"name": "good", "resolves": True, "identity_ok": True, "passage_ok": True, "support_ok": True, "policy_ok": True},
    {"name": "fabricated_source", "resolves": False, "identity_ok": False, "passage_ok": False, "support_ok": False, "policy_ok": False},
    {"name": "wrong_paper", "resolves": True, "identity_ok": False, "passage_ok": False, "support_ok": False, "policy_ok": False},
    {"name": "strength_inflated_37_for_17", "resolves": True, "identity_ok": True, "passage_ok": True, "support_ok": False, "policy_ok": False},
    {"name": "inadmissible_blog_for_clinic", "resolves": True, "identity_ok": True, "passage_ok": True, "support_ok": True, "policy_ok": False},
]

print(f"{'citation':>28} | {' '.join(f'{s:>8}' for s in stages)}")
results = {}
for c in cites:
    r = check_chain(c)
    results[c["name"]] = r
    print(f"{c['name']:>28} | {' '.join(f'{v:>8}' for v in r)}")


In [ ]:
assert results["good"] == ["PASS", "PASS", "PASS", "PASS", "PASS"]
assert results["fabricated_source"][0] == "FAIL"  # never resolves
assert results["wrong_paper"][:2] == ["PASS", "FAIL"]  # resolves, identity wrong
assert results["strength_inflated_37_for_17"][:4] == ["PASS", "PASS", "PASS", "FAIL"]
assert results["inadmissible_blog_for_clinic"] == ["PASS", "PASS", "PASS", "PASS", "FAIL"]
fails_at = {k: v.index("FAIL") for k, v in results.items() if "FAIL" in v}
assert len(set(fails_at.values())) == 4  # four distinct failure links across fixtures
print("chain separation confirmed: each bad citation fails at a different link", fails_at)


## What we earned

- Comparing the unresolved string cannot decide truth: the same sentence verifies as SUPPORTS under the completed-acquisition reading and INSUFFICIENT under the announced reading. Normalize before verifying.
- The citation chain separates five links: four bad fixtures fail at four different stages (resolution, identity, support-strength, admissibility), so a single citation score cannot diagnose them.

Next: Chapter 4 — How Do You Measure a Hallucination?, which fills the verification state with typed sensor outputs and their blind spots.
